In [8]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/laptop_pricing_dataset_mod2.csv"

In [9]:
import micropip
await micropip.install("pyodide-http")

import pyodide_http
pyodide_http.patch_all()  # patches requests/urllib to work in Pyodide

import requests
r = requests.get(URL)
with open("dataset3.csv", "wb") as f:
    f.write(r.content)

In [10]:
import pandas as pd
file_name = "dataset3.csv"
# Path to the CSV file
file_path = file_name

# Read the CSV into a DataFrame. By default, header=0, so the first row is treated as column names.
df = pd.read_csv(file_path)

# Optional: display basic information about the loaded DataFrame
print(df.head())
print('Rows:', len(df), 'Columns:', len(df.columns))

   Unnamed: 0.1  Unnamed: 0 Manufacturer  Category  GPU  OS  CPU_core  \
0             0           0         Acer         4    2   1         5   
1             1           1         Dell         3    1   1         3   
2             2           2         Dell         3    1   1         7   
3             3           3         Dell         4    2   1         5   
4             4           4           HP         4    2   1         7   

   Screen_Size_inch  CPU_frequency  RAM_GB  Storage_GB_SSD  Weight_pounds  \
0              14.0       0.551724       8             256        3.52800   
1              15.6       0.689655       4             256        4.85100   
2              15.6       0.931034       8             256        4.85100   
3              13.3       0.551724       8             128        2.69010   
4              15.6       0.620690       8             256        4.21155   

   Price Price-binned  Screen-Full_HD  Screen-IPS_panel  
0    978          Low               0   

In [15]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Configuration: adjust these for your data
file_path = file_name  # CSV with header row
feature_col = "feature"  # source variable (X)
target_col = "target"    # target variable (y)

# Load data
df = pd.read_csv(file_path, header=0)

# Prepare features and target
X = df[["Price"]]
y = df["CPU_frequency"]

# Train model
model = LinearRegression()
model.fit(X, y)

# Predict and evaluate
y_pred = model.predict(X)
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

# Output results
print("MSE:", mse)
print("R^2:", r2)

MSE: 0.01734537560664534
R^2: 0.1344436321024326


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Read the CSV file into a DataFrame
file_path = file_name
df = pd.read_csv(file_path, header=0)

# Define the source variables and target variable
source_columns = ["CPU_frequency", "RAM_GB", "Storage_GB_SSD","CPU_core","OS", "GPU", "Category"]
target_column = "Price"

# Keep only the required columns and remove rows with missing values
data = df[source_columns + [target_column]].dropna()

# Define input features and target
X = data[source_columns]
y = data[target_column]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create and train the multiple linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions on the test data
y_pred = model.predict(X_test)

# Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Display the results
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f}")

# Optional: display the model equation parameters
print(f"Intercept: {model.intercept_:.4f}")

for feature, coefficient in zip(source_columns, model.coef_):
    print(f"Coefficient for {feature}: {coefficient:.4f}")

Mean Squared Error (MSE): 168575.6204
R² Score: 0.2685
Intercept: -806.6522
Coefficient for CPU_frequency: 1023.7116
Coefficient for RAM_GB: 98.7473
Coefficient for Storage_GB_SSD: 0.0253
Coefficient for CPU_core: 75.7224
Coefficient for OS: -499.8707
Coefficient for GPU: 130.1097
Coefficient for Category: 157.6787


In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load the CSV file
file_path = file_name
df = pd.read_csv(file_path, header=0)

# Specify the source and target columns
source_column = "CPU_frequency"
target_column = "Price"

# Select the required columns and remove missing values
data = df[[source_column, target_column]].dropna()

# Source variable must be a two-dimensional array/DataFrame
X = data[[source_column]]
y = data[target_column]

# Use the same train/test split for every model
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Polynomial degrees to evaluate
degrees = [2, 3, 5]

results = []
models = {}

for degree in degrees:
    # Create a polynomial regression model
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression()
    )

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate performance metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Store the model and its results
    models[degree] = model
    results.append({
        "Polynomial Degree": degree,
        "MSE": mse,
        "R2": r2
    })

# Display the results
results_df = pd.DataFrame(results)

print("Model Performance:")
print(results_df.to_string(index=False))

# Compare the models
best_mse_degree = results_df.loc[results_df["MSE"].idxmin(), "Polynomial Degree"]
best_r2_degree = results_df.loc[results_df["R2"].idxmax(), "Polynomial Degree"]

print("\nModel Comparison:")
print(f"Best model based on lowest MSE: Polynomial degree {best_mse_degree}")
print(f"Best model based on highest R²: Polynomial degree {best_r2_degree}")



Model Performance:
 Polynomial Degree           MSE       R2
                 2 196263.561458 0.148398
                 3 205918.030208 0.106507
                 5 207335.703610 0.100356

Model Comparison:
Best model based on lowest MSE: Polynomial degree 2
Best model based on highest R²: Polynomial degree 2


In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load the data
file_path = file_name
df = pd.read_csv(file_path, header=0)

# Define multiple source features and one target variable
source_columns = ["CPU_frequency", "RAM_GB", "Storage_GB_SSD", "CPU_core", "OS", "GPU", "Category"]
target_column = "Price"

# Select the required columns and remove rows with missing values
data = df[source_columns + [target_column]].dropna()

X = data[source_columns]
y = data[target_column]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create the pipeline:
# 1. Scale the input features
# 2. Generate polynomial features
# 3. Train a linear regression model
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("polynomial_features", PolynomialFeatures(
        degree=2,
        include_bias=False
    )),
    ("linear_regression", LinearRegression())
])

# Train the pipeline
pipeline.fit(X_train, y_train)

# Make predictions
y_pred = pipeline.predict(X_test)

# Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Display results
print("Polynomial Regression Pipeline Results")
print("---------------------------------------")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f}")

Polynomial Regression Pipeline Results
---------------------------------------
Mean Squared Error (MSE): 241404.5074
R² Score: -0.0475


In [21]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# --------------------------------------------------
# 1. Load the data
# --------------------------------------------------
file_path = file_name
df = pd.read_csv(file_path, header=0)

# Specify the attributes to use as input features
source_columns = ["CPU_frequency", "RAM_GB", "Storage_GB_SSD", "CPU_core", "OS", "GPU", "Category"]

# Specify the target attribute
target_column = "Price"

# Select the required columns and remove missing values
data = df[source_columns + [target_column]].dropna()

X = data[source_columns]
y = data[target_column]

# --------------------------------------------------
# 2. Split the data into training and testing sets
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# --------------------------------------------------
# 3. Create the pipeline
# --------------------------------------------------
# The pipeline:
#   - Generates polynomial features
#   - Scales the generated features
#   - Trains a Ridge regression model
pipeline = Pipeline([
    ("polynomial_features", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# --------------------------------------------------
# 4. Define the Grid Search parameters
# --------------------------------------------------
# Alpha controls the strength of Ridge regularization.
# The polynomial degree can also be tested by Grid Search.
parameter_grid = {
    "polynomial_features__degree": [2, 3],
    "ridge__alpha": [0.0001, 0.01, 0.1, 1, 1, 10]
}

# --------------------------------------------------
# 5. Perform Grid Search with cross-validation
# --------------------------------------------------
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=parameter_grid,
    cv=5,                       # Five-fold cross-validation
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    refit=True
)

grid_search.fit(X_train, y_train)

# --------------------------------------------------
# 6. Evaluate the best model on the test set
# --------------------------------------------------
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# --------------------------------------------------
# 7. Display the results
# --------------------------------------------------
print("Best Parameters:")
print(grid_search.best_params_)

print(f"\nBest Cross-Validation MSE: {-grid_search.best_score_:.4f}")
print(f"Test Mean Squared Error (MSE): {mse:.4f}")
print(f"Test R² Score: {r2:.4f}")

Best Parameters:
{'polynomial_features__degree': 3, 'ridge__alpha': 10}

Best Cross-Validation MSE: 140360.3352
Test Mean Squared Error (MSE): 219068.1516
Test R² Score: 0.0494
